In [9]:
import Pkg

project_dir = pwd()
isfile(joinpath(project_dir, "Project.toml")) || error("Start this notebook in the data_generalization directory")
Pkg.activate(project_dir)


  Activating project at `~/Projects/ml-opf-bench/ML-OPF-Bench/data_generalization`


In [10]:
module ACConstraints

using PowerModels, DataFrames, CSV

function export_ac_constraints(case_path::String, output_dir::String)
    mkpath(output_dir)
    case_name = splitext(basename(case_path))[1]

    data = PowerModels.parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    println("Parsed case: $(case_name)")

    base_mva = ref[:baseMVA]
    df_base_mva = DataFrame(parameter=["baseMVA"], value=[base_mva])
    CSV.write(joinpath(output_dir, "$(case_name)_base_mva.csv"), df_base_mva)

    bus_ids = sort(collect(keys(ref[:bus])))

    load_pd = Dict(id => 0.0 for id in bus_ids)
    load_qd = Dict(id => 0.0 for id in bus_ids)

    for (_, load) in ref[:load]
        if get(load, "status", 1) == 1
            b = load["load_bus"]
            load_pd[b] += get(load, "pd", 0.0)
            load_qd[b] += get(load, "qd", 0.0)
        end
    end

    shunt_gs = Dict(id => 0.0 for id in bus_ids)
    shunt_bs = Dict(id => 0.0 for id in bus_ids)

    for (_, sh) in get(ref, :shunt, Dict())
        if get(sh, "status", 1) == 1
            b = sh["shunt_bus"]
            shunt_gs[b] += get(sh, "gs", 0.0)
            shunt_bs[b] += get(sh, "bs", 0.0)
        end
    end

    bus_data = DataFrame(
        bus_id  = Int[],
        type    = Int[],
        pd_pu   = Float64[],
        qd_pu   = Float64[],
        vmin_pu = Float64[],
        vmax_pu = Float64[],
        vm_pu   = Float64[],
        va_rad  = Float64[],
        base_kv = Float64[],
        gs_pu   = Float64[],
        bs_pu   = Float64[]
    )

    for id in bus_ids
        bus = ref[:bus][id]
        push!(bus_data, [
            id,
            bus["bus_type"],
            load_pd[id],
            load_qd[id],
            bus["vmin"],
            bus["vmax"],
            get(bus, "vm", 1.0),
            get(bus, "va", 0.0),
            get(bus, "base_kv", 110.0),
            shunt_gs[id],
            shunt_bs[id]
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_bus_data.csv"), bus_data)

    gen_ids = sort(collect(keys(ref[:gen])))
    gen_data = DataFrame(
        gen_id      = Int[],
        bus_id      = Int[],
        pg_min_pu   = Float64[],
        pg_max_pu   = Float64[],
        qg_min_pu   = Float64[],
        qg_max_pu   = Float64[],
        vg_pu       = Float64[],
        cost_c2     = Float64[],
        cost_c1     = Float64[],
        cost_c0     = Float64[]
    )

    for id in gen_ids
        g = ref[:gen][id]

        coeffs = get(g, "cost", Float64[])
        c2, c1, c0 = 0.0, 0.0, 0.0
        if length(coeffs) == 3
            c2, c1, c0 = coeffs[1], coeffs[2], coeffs[3]
        elseif length(coeffs) == 2
            c1, c0 = coeffs[1], coeffs[2]
        end

        push!(gen_data, [
            id,
            g["gen_bus"],
            g["pmin"],
            g["pmax"],
            g["qmin"],
            g["qmax"],
            g["vg"],
            c2, c1, c0
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_gen_data.csv"), gen_data)

    n_buses = length(bus_ids)
    n_gen = length(gen_ids)
    bus_id_to_idx = Dict(bus_id => idx for (idx, bus_id) in enumerate(bus_ids))

    bus_gen_matrix = zeros(Int, n_buses, n_gen)
    for (gen_idx, gen_id) in enumerate(gen_ids)
        gen_bus_id = ref[:gen][gen_id]["gen_bus"]
        bus_idx = bus_id_to_idx[gen_bus_id]
        bus_gen_matrix[bus_idx, gen_idx] = 1
    end

    gen_col_names = ["gen_$i" for i in 1:n_gen]
    bus_gen_df = DataFrame(bus_gen_matrix, gen_col_names)
    insertcols!(bus_gen_df, 1, :bus_id => bus_ids)
    CSV.write(joinpath(output_dir, "$(case_name)_bus_gen_map.csv"), bus_gen_df)

    branch_ids = sort(collect(keys(ref[:branch])))
    branch_data = DataFrame(
        branch_id = Int[],
        f_bus     = Int[],
        t_bus     = Int[],
        r_pu      = Float64[],
        x_pu      = Float64[],
        b_pu      = Float64[],
        rate_a_pu = Float64[],
        tap_ratio = Float64[],
        shift_rad = Float64[],
        angmin_rad = Float64[],
        angmax_rad = Float64[]
    )

    for id in branch_ids
        br = ref[:branch][id]

        rate_a_raw = get(br, "rate_a", Inf)

        rate_a_val = isfinite(rate_a_raw) ? rate_a_raw : 0.0

        push!(branch_data, [
            id,
            br["f_bus"], br["t_bus"],
            br["br_r"], br["br_x"],
            get(br, "b_fr", 0.0) + get(br, "b_to", 0.0),
            rate_a_val,
            get(br, "tap", 1.0),
            get(br, "shift", 0.0),
            get(br, "angmin", -pi),
            get(br, "angmax", pi)
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_branch_data.csv"), branch_data)

    slack_buses = DataFrame(bus_id = collect(keys(ref[:ref_buses])))
    CSV.write(joinpath(output_dir, "$(case_name)_slack_buses.csv"), slack_buses)

    println("Done (p.u. units). Files saved to: $(output_dir)")
end

end


Main.ACConstraints

In [13]:
case_file = joinpath(project_dir, "..", "test_systems", "typ", "pglib_opf_case300_ieee.m")
output_dir = joinpath(project_dir, "..", "ac_dataset", "acopf_constraints", "case300")


"/Users/xinyiliu/Projects/ml-opf-bench/ML-OPF-Bench/data_generalization/../ac_dataset/acopf_constraints/case300"

In [14]:
ACConstraints.export_ac_constraints(case_file, output_dir)


[ PowerModels | Info]: removing 1 cost terms from generator 32: [3101.6664, 0.0]
[ PowerModels | Info]: removing 1 cost terms from generator 29: [2727.8538000000003, 0.0]
[ PowerModels | Info]: removing 3 cost terms from generator 1: Float64[]
[ PowerModels | Info]: removing 1 cost terms from generator 54: [10398.7496, 0.0]
[ PowerModels | Info]: removing 3 cost terms from generator 2: Float64[]
[ PowerModels | Info]: removing 1 cost terms from generator 41: [3965.7688, 0.0]
[ PowerModels | Info]: removing 3 cost terms from generator 65: Float64[]
[ PowerModels | Info]: removing 1 cost terms from generator 51: [3512.6651, 0.0]
[ PowerModels | Info]: removing 1 cost terms from generator 53: [2999.2014999999997, 0.0]
[ PowerModels | Info]: removing 1 cost terms from generator 27: [2839.1517, 0.0]
[ PowerModels | Info]: removing 1 cost terms from generator 42: [4122.3312, 0.0]
[ PowerModels | Info]: removing 1 cost terms from generator 33: [4972.1799, 0.0]
[ PowerModels | Info]: removing 